# 🧮 Notebook 02: Math Foundations for Transformers

Every transformer — GPT, LLaMA, Gemma — is built from a small set of math operations. This notebook breaks each one down step by step in MLX so you can see exactly what happens inside the model.

**What we'll cover:**

| Operation | Why It Matters |
|---|---|
| **Dot product** | Core of attention — measures similarity between tokens |
| **Matrix multiplication** | Every linear layer, every projection |
| **Softmax** | Turns raw scores into probabilities |
| **Temperature & sampling** | Controls creativity vs determinism at generation time |
| **Cross-entropy loss** | The training signal — how wrong are our predictions? |

**Prerequisites:** Notebook 00 (environment) and Notebook 01 (MLX basics).

## ✅ Environment Validation

Let's confirm MLX, Metal, and Python are ready before diving into the math.

In [1]:
from utils.checks import validate_environment, print_environment_report

results = print_environment_report()

# Verify critical requirements for this notebook
assert results["mlx_available"], "MLX is required — install with: pip install mlx"
assert results["metal_available"], "Metal GPU is required for GPU benchmarks"
print("\n🧮 Ready for math foundations!")

  Environment Validation Report
  Platform : macOS-26.4.1-arm64-arm-64bit-Mach-O
  Chip     : Apple M4 Pro
  Python   : 3.13.13  ✅
  MLX      : 0.31.1  ✅
  Metal GPU: available  ✅
  Memory   : 48.0 GB  ✅

🧮 Ready for math foundations!


---
## 📐 Overview: The Math You Actually Need

You don't need a math PhD to understand transformers. Here are the five operations that make up ~95% of what happens inside every LLM:

1. **Dot product** — "How similar are these two vectors?" → Used in attention to compute how much token A should attend to token B.
2. **Matrix multiplication** — "Transform a batch of vectors at once." → Every linear layer, every Q/K/V projection.
3. **Softmax** — "Turn arbitrary numbers into a probability distribution." → Converts attention scores into weights that sum to 1.
4. **Temperature & sampling** — "How creative should the model be?" → Controls the sharpness of the output distribution at generation time.
5. **Cross-entropy loss** — "How wrong was the prediction?" → The training signal that tells the model which direction to update.

Let's build each one from scratch in MLX.

---
## 🔢 Dot Product

The dot product is the most fundamental operation in attention. It answers: **"How similar are these two vectors?"**

Given two vectors $a$ and $b$ of the same length, the dot product is:

$$a \cdot b = \sum_{i} a_i \times b_i$$

In a transformer, this computes how much one token's query matches another token's key.

### 💡 What is a dot product, intuitively?

Think of it like a **compatibility score on a dating app**. Each person has a profile with numeric preferences (loves hiking: 9/10, loves cooking: 3/10, etc.). The dot product between two profiles tells you how compatible they are — it multiplies matching preferences and adds them up. High score = great match, low score = mismatch.

In a transformer, each token has a "profile" (its embedding vector). The dot product between two tokens' vectors tells the model: *"How much should I pay attention to this other token?"* That's literally what attention computes.

### Approach 1: Element-wise Multiply Then Sum

The most explicit way to compute a dot product. We multiply corresponding elements, then sum the results. This is exactly what happens under the hood.

In [2]:
import mlx.core as mx
import numpy as np

# Two 4-dimensional vectors (think: 4-dim token embeddings)
a = mx.array([1.0, 2.0, 3.0, 4.0])  # shape: (4,)
b = mx.array([5.0, 6.0, 7.0, 8.0])  # shape: (4,)

# Step 1: Element-wise multiplication
elementwise = a * b  # shape: (4,) — each element multiplied
mx.eval(elementwise)
print(f"a           = {a.tolist()}")
print(f"b           = {b.tolist()}")
print(f"a * b       = {elementwise.tolist()}")
print(f"             {1*5} + {2*6} + {3*7} + {4*8}")

# Step 2: Sum to get scalar
dot_manual = mx.sum(elementwise)  # shape: () — scalar
mx.eval(dot_manual)
print(f"sum(a * b)  = {dot_manual.item()}")

a           = [1.0, 2.0, 3.0, 4.0]
b           = [5.0, 6.0, 7.0, 8.0]
a * b       = [5.0, 12.0, 21.0, 32.0]
             5 + 12 + 21 + 32
sum(a * b)  = 70.0


### Approach 2: The `@` Operator

MLX (like NumPy and PyTorch) supports the `@` operator for matrix/vector products. For 1-D vectors, `a @ b` computes the dot product directly. This is what you'll see in real transformer code.

In [3]:
# The @ operator: clean and fast
dot_at = a @ b  # shape: () — scalar
mx.eval(dot_at)
print(f"a @ b = {dot_at.item()}")
print(f"Manual match: {dot_manual.item() == dot_at.item()}")

# In attention, this becomes: score = query @ key
# Higher dot product = more similar vectors = higher attention weight
print(f"\n💡 In attention: score(token_A, token_B) = query_A @ key_B = {dot_at.item()}")

a @ b = 70.0
Manual match: True

💡 In attention: score(token_A, token_B) = query_A @ key_B = 70.0


### ⚠️ Handling Common Errors

When working with ML code, errors are normal and expected. Here's a pattern for handling them gracefully — if something goes wrong, you get a helpful message instead of a crash.

In [4]:
# 💡 Error handling pattern — use this when operations might fail
try:
    # This is where your ML code goes
    import mlx.core as mx
    test = mx.array([1.0, 2.0, 3.0])
    result = mx.sum(test)
    mx.eval(result)
    print(f'✅ Success! Result: {result.item()}')
except Exception as e:
    print(f'❌ Error: {e}')
    print('💡 Tip: Check that MLX is installed and your inputs are valid')

✅ Success! Result: 6.0


### Cosine Similarity

Raw dot products are affected by vector magnitude — longer vectors get higher scores regardless of direction. **Cosine similarity** normalizes by vector lengths, giving a pure measure of directional similarity in $[-1, 1]$:

$$\text{cosine\_sim}(a, b) = \frac{a \cdot b}{\|a\| \times \|b\|}$$

- **1.0** = vectors point the same direction (identical meaning)
- **0.0** = vectors are orthogonal (unrelated)
- **-1.0** = vectors point opposite directions

In [5]:
def cosine_similarity(a: mx.array, b: mx.array) -> mx.array:
    """Compute cosine similarity between two vectors."""
    dot = mx.sum(a * b)                          # a · b
    norm_a = mx.sqrt(mx.sum(a * a))              # ||a||
    norm_b = mx.sqrt(mx.sum(b * b))              # ||b||
    return dot / (norm_a * norm_b)               # normalized similarity

# Same direction (similar tokens)
v1 = mx.array([1.0, 0.0, 0.0])
v2 = mx.array([1.0, 0.0, 0.0])
sim1 = cosine_similarity(v1, v2)
mx.eval(sim1)
print(f"Same direction:     cosine_sim = {sim1.item():.4f}")

# Orthogonal (unrelated tokens)
v3 = mx.array([0.0, 1.0, 0.0])
sim2 = cosine_similarity(v1, v3)
mx.eval(sim2)
print(f"Orthogonal:         cosine_sim = {sim2.item():.4f}")

# Opposite direction
v4 = mx.array([-1.0, 0.0, 0.0])
sim3 = cosine_similarity(v1, v4)
mx.eval(sim3)
print(f"Opposite direction: cosine_sim = {sim3.item():.4f}")

# Realistic example: similar vs different embeddings
emb_cat = mx.array([0.8, 0.1, 0.9, 0.2])   # "cat" embedding
emb_dog = mx.array([0.7, 0.2, 0.85, 0.15]) # "dog" embedding (similar animal)
emb_car = mx.array([0.1, 0.9, 0.1, 0.8])   # "car" embedding (very different)

sim_cat_dog = cosine_similarity(emb_cat, emb_dog)
sim_cat_car = cosine_similarity(emb_cat, emb_car)
mx.eval(sim_cat_dog, sim_cat_car)
print(f"\ncat vs dog: {sim_cat_dog.item():.4f}  (similar — both animals)")
print(f"cat vs car: {sim_cat_car.item():.4f}  (different — animal vs vehicle)")

Same direction:     cosine_sim = 1.0000
Orthogonal:         cosine_sim = 0.0000
Opposite direction: cosine_sim = -1.0000

cat vs dog: 0.9943  (similar — both animals)
cat vs car: 0.2828  (different — animal vs vehicle)


### 🎯 Connection to Attention: "How Much Should Token A Attend to Token B?"

In a transformer's attention mechanism, each token produces a **query** vector ("what am I looking for?") and a **key** vector ("what do I contain?"). The dot product `query_A @ key_B` measures how relevant token B is to token A.

```
"The cat sat on the mat"
       ↓
  query("sat") @ key("cat") = high score  → "sat" attends strongly to "cat"
  query("sat") @ key("the") = low score   → "sat" barely attends to "the"
```

💡 **Insight:** The entire attention mechanism is just a batched dot product followed by softmax. Everything else is bookkeeping.

🎯 **Interview tip:** When asked "what does attention compute?", start with: "It's a weighted average of value vectors, where the weights come from dot-product similarity between queries and keys."

---
## 📊 Matrix Multiplication

Every linear layer in a transformer is a matrix multiplication. When a batch of token embeddings passes through a projection layer, this is what happens under the hood.

For matrices $A$ of shape $(m, k)$ and $B$ of shape $(k, n)$:

$$C = A \times B \quad \text{shape: } (m, n)$$

Each element $C_{ij} = \sum_{p} A_{ip} \times B_{pj}$ — it's a dot product of row $i$ of $A$ with column $j$ of $B$.

### Token Embeddings Through a Linear Layer

Let's trace a batch of token embeddings through a linear projection, annotating shapes at every step. This is exactly what happens when computing Q, K, or V in attention.

In [6]:
import mlx.core as mx

# Simulate a batch of token embeddings
batch_size = 2
seq_len = 5
d_model = 8   # embedding dimension
d_out = 6     # output dimension (e.g., d_head for Q/K/V projection)

# Input: batch of token embeddings
x = mx.random.normal(shape=(batch_size, seq_len, d_model))  # shape: (2, 5, 8)
print(f"Input x shape:  {x.shape}  ← (batch, seq_len, d_model)")

# Weight matrix (like W_q, W_k, or W_v in attention)
W = mx.random.normal(shape=(d_model, d_out))  # shape: (8, 6)
print(f"Weight W shape: {W.shape}  ← (d_model, d_out)")

# Bias vector
b = mx.zeros(shape=(d_out,))  # shape: (6,)
print(f"Bias b shape:   {b.shape}    ← (d_out,)")

# Matrix multiply: each token embedding (d_model,) @ W (d_model, d_out) → (d_out,)
# MLX broadcasts over batch and seq_len dimensions automatically
output = x @ W + b  # shape: (2, 5, 6)
mx.eval(output)
print(f"\nOutput shape:   {output.shape}  ← (batch, seq_len, d_out)")
print(f"\nShape journey:  (2, 5, 8) @ (8, 6) + (6,) → (2, 5, 6)")
print(f"                 batch×seq×d_model  @  d_model×d_out  →  batch×seq×d_out")

# Verify: pick one token and do it manually
token_0_0 = x[0, 0, :]  # shape: (8,) — first token of first batch
manual = token_0_0 @ W + b  # shape: (6,)
mx.eval(manual)
print(f"\nManual single token: {manual.tolist()[:3]}...")
print(f"Batched result[0,0]: {output[0, 0, :].tolist()[:3]}...")
print(f"Match: {mx.allclose(manual, output[0, 0, :]).item()}")

Input x shape:  (2, 5, 8)  ← (batch, seq_len, d_model)
Weight W shape: (8, 6)  ← (d_model, d_out)
Bias b shape:   (6,)    ← (d_out,)

Output shape:   (2, 5, 6)  ← (batch, seq_len, d_out)

Shape journey:  (2, 5, 8) @ (8, 6) + (6,) → (2, 5, 6)
                 batch×seq×d_model  @  d_model×d_out  →  batch×seq×d_out

Manual single token: [-4.581515312194824, -0.9902368187904358, 5.571959972381592]...
Batched result[0,0]: [-4.581515789031982, -0.9902368187904358, 5.57196044921875]...
Match: True


### ⚡ GPU Benchmark: Matrix Multiplication at Scale

Matrix multiplication is the most compute-intensive operation in transformers. Let's benchmark the Apple Silicon GPU across different matrix sizes and calculate GFLOPS (billions of floating-point operations per second).

For multiplying two $(N, N)$ matrices, the number of FLOPs is $2N^3$ (each of $N^2$ output elements requires $N$ multiplies and $N-1$ adds).

In [7]:
from utils.benchmark import time_function

def matmul_benchmark(N: int) -> mx.array:
    """Multiply two NxN random matrices."""
    A = mx.random.normal(shape=(N, N))
    B = mx.random.normal(shape=(N, N))
    mx.eval(A, B)  # ensure inputs are materialized
    return A @ B

sizes = [256, 512, 1024, 2048, 4096]
print(f"{'Size':>10} {'Time (ms)':>12} {'GFLOPS':>10}")
print("-" * 36)

benchmark_results = []
for N in sizes:
    stats = time_function(matmul_benchmark, N, warmup=2, repeats=5)
    flops = 2 * N**3                          # FLOPs for NxN matmul
    gflops = flops / (stats['mean_ms'] / 1000) / 1e9  # GFLOPS
    print(f"{N:>7}x{N:<4} {stats['mean_ms']:>10.2f}ms {gflops:>9.1f}")
    benchmark_results.append((N, stats['mean_ms'], gflops))

print(f"\n⚡ Peak throughput: {max(r[2] for r in benchmark_results):.1f} GFLOPS")
print(f"💡 Larger matrices = better GPU utilization (more parallelism)")

      Size    Time (ms)     GFLOPS
------------------------------------
    256x256        0.53ms      63.1
    512x512        1.13ms     237.3
   1024x1024       1.92ms    1117.8
   2048x2048       6.12ms    2806.5


   4096x4096      34.35ms    4001.2

⚡ Peak throughput: 4001.2 GFLOPS
💡 Larger matrices = better GPU utilization (more parallelism)


🎯 **Interview tip:** When asked about transformer compute costs, remember: a forward pass through a model with $N$ parameters on a sequence of length $S$ costs roughly $2NS$ FLOPs. The backward pass is ~2× that. Matrix multiplication dominates the compute budget.

⚡ **Performance note:** Apple Silicon's GPU gets better utilization with larger matrices because there's more work to distribute across the GPU cores. This is why batching matters — small matrices underutilize the hardware.

### 🔢 Quick Math Primer: What is `e`? What is `log`?

Before we dive into softmax, you need two math concepts. Don't worry — they're simpler than they look.

**The number `e` ≈ 2.718** — It's a special constant (like π ≈ 3.14159) that shows up everywhere in math and nature. The key property: `e^x` grows faster and faster as x increases. Think of it as "compound interest on steroids."

| x | e^x | Intuition |
|---|-----|-----------|
| 0 | 1.0 | e^0 is always 1 |
| 1 | 2.7 | About 2.7× growth |
| 2 | 7.4 | Much bigger! |
| 5 | 148.4 | Exploding! |
| -2 | 0.14 | Negative x → small positive number (never zero) |

**The `log` function** — It's the REVERSE of `e^x`. If e^2 ≈ 7.4, then log(7.4) ≈ 2. Think of it as asking "what power do I raise e to, to get this number?"

| x | log(x) | Intuition |
|---|--------|-----------|
| 1 | 0 | log(1) = 0 always |
| 2.7 | ~1 | log(e) = 1 |
| 7.4 | ~2 | log(e²) = 2 |
| 0.5 | -0.7 | Numbers < 1 have negative log |

**Why do we care?** Softmax uses `e^x` to turn scores into probabilities. Cross-entropy loss uses `log` to measure how surprised the model is. These two functions are the mathematical backbone of every LLM.

---

### 🎯 Interview Question nb02-q01  ·  [warmup]  ·  mle

**Q:** What is the difference between numerator-layout and denominator-layout matrix calculus, and which convention does PyTorch / MLX autograd actually use?

<details>
<summary>Key points in a strong answer</summary>

- Numerator layout: ∂y/∂x has shape (dim(y), dim(x)) — the numerator's dimensions come first.
- Denominator layout: ∂y/∂x has shape (dim(x), dim(y)) — the denominator's dimensions come first. The two layouts are transposes of each other.
- Autograd frameworks (PyTorch, MLX, JAX) expose `grad(y) w.r.t. x` as having the SAME shape as `x`, not a Jacobian — this matches reverse-mode AD's vector-Jacobian-product semantics.
- For a scalar loss `L` and parameter tensor `W` with shape `(m, n)`, `∂L/∂W` also has shape `(m, n)` so SGD can write `W -= lr * grad`.
- Implication: when you write a matrix-calc derivation on paper, you must match your convention to the framework's or you'll get a transposed update rule — a classic source of off-by-transpose bugs.
</details>

> ⚠️ **Trap:** Claiming 'autograd returns the Jacobian' — it returns a vector-Jacobian product shaped like the parameter, never a full dense Jacobian (which for a 1B-param model would be a ~4 EB matrix).
>
> 📚 **References:** https://en.wikipedia.org/wiki/Matrix_calculus, https://pytorch.org/docs/stable/notes/autograd.html

---
## 🌟 Softmax

Softmax converts a vector of arbitrary real numbers ("logits") into a probability distribution where all values are in $[0, 1]$ and sum to 1:

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

In transformers, softmax is used in two critical places:
1. **Attention weights** — turning raw Q·K scores into a probability distribution over tokens
2. **Output layer** — turning logits into next-token probabilities

### 💡 What is softmax, intuitively?

Imagine you're at a restaurant and you rate each dish on how appealing it looks: pasta gets 8, salad gets 3, soup gets 5. Softmax turns those raw ratings into **"what's the probability I'll order each dish?"** — maybe 73% pasta, 5% salad, 22% soup. The highest-rated dish gets the lion's share of probability, but the others still have a chance.

In a transformer, softmax does the same thing with attention scores. After computing how relevant each token is (via dot products), softmax converts those raw scores into a probability distribution: *"What fraction of my attention should go to each token?"* The scores always sum to 1, so it's a proper budget allocation.

### Manual Softmax with Numerical Stability

Naive softmax can overflow: $e^{1000}$ is infinity in float32. The fix is simple — subtract the maximum value first. This doesn't change the result (it cancels out) but keeps numbers in a safe range.

$$\text{softmax}(x_i) = \frac{e^{x_i - \max(x)}}{\sum_j e^{x_j - \max(x)}}$$

⚠️ **Pitfall:** Forgetting the max-subtraction trick is a classic bug. It works fine on small numbers but explodes on real model logits.

In [8]:
import mlx.core as mx
import numpy as np

def softmax_manual(logits: mx.array) -> mx.array:
    """Numerically stable softmax implementation."""
    # Step 1: Subtract max for numerical stability
    max_val = mx.max(logits, axis=-1, keepdims=True)  # shape: (..., 1)
    shifted = logits - max_val                          # shape: same as logits
    
    # Step 2: Exponentiate
    exp_vals = mx.exp(shifted)                          # shape: same as logits
    
    # Step 3: Normalize (divide by sum)
    sum_exp = mx.sum(exp_vals, axis=-1, keepdims=True)  # shape: (..., 1)
    probs = exp_vals / sum_exp                          # shape: same as logits
    
    return probs

# Test with simple logits
logits = mx.array([2.0, 1.0, 0.1])
probs = softmax_manual(logits)
mx.eval(probs)

print(f"Logits:        {logits.tolist()}")
print(f"Probabilities: {[f'{p:.4f}' for p in probs.tolist()]}")
print(f"Sum:           {mx.sum(probs).item():.6f}  (should be 1.0)")
print(f"All in [0,1]:  {bool(mx.all(probs >= 0).item() and mx.all(probs <= 1).item())}")

# Test with extreme logits (this is where stability matters!)
print("\n⚠️  Testing with extreme logits (would overflow without max trick):")
extreme_logits = mx.array([1000.0, 999.0, 998.0])
extreme_probs = softmax_manual(extreme_logits)
mx.eval(extreme_probs)
print(f"Logits:        {extreme_logits.tolist()}")
print(f"Probabilities: {[f'{p:.4f}' for p in extreme_probs.tolist()]}")
print(f"Sum:           {mx.sum(extreme_probs).item():.6f}")
print(f"No NaN:        {bool(mx.all(mx.isnan(extreme_probs) == False).item())}")

Logits:        [2.0, 1.0, 0.10000000149011612]
Probabilities: ['0.6590', '0.2424', '0.0986']
Sum:           1.000000  (should be 1.0)
All in [0,1]:  True

⚠️  Testing with extreme logits (would overflow without max trick):
Logits:        [1000.0, 999.0, 998.0]
Probabilities: ['0.6652', '0.2447', '0.0900']
Sum:           1.000000
No NaN:        True


### Comparison with `mx.softmax`

MLX provides a built-in softmax that's optimized and fused on the GPU. Let's verify our manual version matches and compare performance.

In [9]:
# Compare manual vs built-in
logits = mx.random.normal(shape=(4, 10))  # shape: (4, 10) — 4 samples, 10 classes

manual_result = softmax_manual(logits)     # Our implementation
builtin_result = mx.softmax(logits, axis=-1)  # MLX built-in
mx.eval(manual_result, builtin_result)

# Check they match
max_diff = mx.max(mx.abs(manual_result - builtin_result)).item()
print(f"Max difference: {max_diff:.2e}  (should be ~0)")
print(f"Match: {max_diff < 1e-5}")

# Verify properties
print(f"\nRow sums (manual):  {[f'{s:.6f}' for s in mx.sum(manual_result, axis=-1).tolist()]}")
print(f"Row sums (builtin): {[f'{s:.6f}' for s in mx.sum(builtin_result, axis=-1).tolist()]}")
print(f"All values in [0,1]: {bool(mx.all(manual_result >= 0).item())}")

Max difference: 5.96e-08  (should be ~0)
Match: True

Row sums (manual):  ['1.000000', '1.000000', '1.000000', '1.000000']
Row sums (builtin): ['1.000000', '1.000000', '1.000000', '1.000000']
All values in [0,1]: True


### Visualizing Logits vs Probabilities

Let's see how softmax transforms raw logits into a probability distribution. Notice how the largest logit gets the most probability mass, and the differences get amplified.

In [10]:
import matplotlib.pyplot as plt

# Simulate logits for 8 tokens (like attention scores)
tokens = ["The", "cat", "sat", "on", "the", "warm", "soft", "mat"]
logits = mx.array([0.5, 2.8, 1.2, 0.3, 0.1, 1.5, 0.8, 2.1])
probs = mx.softmax(logits, axis=-1)
mx.eval(probs)

logits_np = np.array(logits.tolist())
probs_np = np.array(probs.tolist())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: raw logits
colors = plt.cm.Blues(0.3 + 0.7 * (logits_np - logits_np.min()) / (logits_np.max() - logits_np.min()))
ax1.bar(tokens, logits_np, color=colors, edgecolor='navy', linewidth=0.5)
ax1.set_title("Raw Logits (before softmax)", fontsize=12)
ax1.set_ylabel("Logit value")
ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
for i, v in enumerate(logits_np):
    ax1.text(i, v + 0.05, f"{v:.1f}", ha='center', fontsize=9)

# Right: probabilities after softmax
colors2 = plt.cm.Oranges(0.3 + 0.7 * probs_np / probs_np.max())
ax2.bar(tokens, probs_np, color=colors2, edgecolor='darkorange', linewidth=0.5)
ax2.set_title("Probabilities (after softmax)", fontsize=12)
ax2.set_ylabel("Probability")
for i, v in enumerate(probs_np):
    ax2.text(i, v + 0.005, f"{v:.3f}", ha='center', fontsize=9)

plt.suptitle("🌟 Softmax: Logits → Probabilities", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"💡 'cat' has the highest logit ({logits_np[1]:.1f}) and gets {probs_np[1]:.1%} of the probability.")
print(f"   Softmax amplifies differences — the winner takes more than its 'fair share'.")

💡 'cat' has the highest logit (2.8) and gets 42.4% of the probability.
   Softmax amplifies differences — the winner takes more than its 'fair share'.


/var/folders/gg/dfm95f2x2llgzc8_3fnnxxzc0000gp/T/ipykernel_99189/1322862810.py:32: UserWarning: Glyph 127775 (\N{GLOWING STAR}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/gg/dfm95f2x2llgzc8_3fnnxxzc0000gp/T/ipykernel_99189/1322862810.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


💡 **Insight:** Softmax is a "soft" version of argmax. As logit differences grow, softmax approaches a one-hot vector (all probability on the largest logit). This is why **temperature** matters — it controls how "peaked" the distribution is.

---

### 🎯 Interview Question nb02-q02  ·  [core]  ·  mle, research_engineer

**Q:** The naive softmax `exp(x) / sum(exp(x))` blows up on attention logits near magnitude 50. Derive the max-subtraction / log-sum-exp trick from first principles and explain *exactly* why it is mathematically equivalent.

<details>
<summary>Key points in a strong answer</summary>

- Multiply numerator and denominator by `exp(-c)` for any constant `c` — the ratio is unchanged: softmax(x)_i = exp(x_i - c) / Σ_j exp(x_j - c).
- Pick c = max(x): all shifted inputs are ≤ 0, so every `exp(x_j - c)` is in (0, 1]; no overflow.
- At least one term equals exp(0) = 1, so the denominator is ≥ 1 — zero underflow for the largest entry (the one that matters most).
- log-sum-exp: log(Σ exp(x_i)) = c + log(Σ exp(x_i - c)); same shift argument, lets us compute log-softmax stably.
- Combined `log_softmax(x) = x - logsumexp(x)` avoids computing exp then log (a double-rounding source); most frameworks fuse it.
- Interview tip: ALWAYS reach for this in code — fp16 logits overflow at ~11.0, bf16 at ~88.7; un-shifted softmax is a latent NaN bomb.
</details>

> ⚠️ **Trap:** Shifting by the mean instead of the max — the trick relies on ALL shifted inputs being ≤ 0 so every exp is bounded by 1; mean-shifting leaves the largest entry positive, which can still overflow.
>
> 📚 **References:** https://en.wikipedia.org/wiki/LogSumExp, https://gregorygundersen.com/blog/2020/02/09/log-sum-exp/

---

### 🎯 Interview Question nb02-q03  ·  [core]  ·  mle, research_engineer, systems_engineer

**Q:** Derive the Jacobian of softmax(x) ∈ ℝⁿ w.r.t. x ∈ ℝⁿ from scratch. Why does the diagonal vs off-diagonal structure matter for efficient backprop?

<details>
<summary>Key points in a strong answer</summary>

- Let s = softmax(x), so s_i = exp(x_i) / Σ_k exp(x_k). Use quotient rule for ∂s_i/∂x_j.
- Diagonal (i == j): ∂s_i/∂x_i = s_i(1 - s_i).
- Off-diagonal (i ≠ j): ∂s_i/∂x_j = -s_i · s_j.
- Compact form: J = diag(s) - s · sᵀ  (an n × n rank-(n-1) matrix).
- Key property: J is symmetric (since s_i·s_j == s_j·s_i) and singular (sum of each row is 0 because Σ s_k = 1 is invariant to a uniform shift in x).
- Efficient backprop: you almost NEVER materialize J. Instead use the identity (J · v)_i = s_i · (v_i - sᵀv) — linear in n, not quadratic.
</details>

> ⚠️ **Trap:** Writing the Jacobian as J = diag(s) - diag(s²); the off-diagonal piece is the OUTER PRODUCT s·sᵀ, not a diagonal elementwise square.
>
> 📚 **References:** https://eli.thegreenplace.net/2016/the-softmax-function-and-its-derivative

---

### 🎯 Interview Question nb02-q07  ·  [research]  ·  research_engineer, systems_engineer

**Q:** Flash-Attention computes softmax 'online' — one tile at a time, without materializing the full attention matrix. Derive the tile-merge formula for log-sum-exp and explain why this is the crux that makes FA-2 fp16-safe.

<details>
<summary>Key points in a strong answer</summary>

- Given two partial running stats (m_a, ℓ_a) and (m_b, ℓ_b) — running max and sum-of-exp relative to that max — the merged (m, ℓ) is: m = max(m_a, m_b); ℓ = exp(m_a - m)·ℓ_a + exp(m_b - m)·ℓ_b.
- At every merge you ADJUST the previously-accumulated ℓ by exp(old_max − new_max) ≤ 1, so no rescale ever grows.
- This keeps every intermediate bounded in (0, ℓ·e], i.e. representable in fp16 (max ~6.5e4) even when raw logits exceed the fp16 exponent range.
- Output `O` tiles follow the same correction: O_new = O_old · exp(m_old - m) + ΔO_new; the rescale ratio is shared with the ℓ update — one extra multiply per tile.
- Memory win: FA-2 never stores the (N × N) softmax matrix — only O(N · d_head) in SRAM; wall-clock win follows because HBM bandwidth was the bottleneck.
- fp16-safety is NOT automatic — DeepSeek-V3 and Gemma still promote the softmax accumulator to fp32 even inside FA-3 tiles because bf16/fp16 precision at ℓ > 10⁴ degrades the tail of the distribution.
</details>

> ⚠️ **Trap:** Assuming 'fp16 softmax = safe softmax' — tile merging keeps range OK, but precision on the long-tail probabilities (where temperature/sampling actually matters) requires an fp32 accumulator.
>
> 📚 **References:** https://arxiv.org/abs/2205.14135, https://arxiv.org/abs/2307.08691, https://tridao.me/publications/flash2/flash2.pdf

---

### 🧑‍💻 Whiteboard Challenge: Numerically-stable log-softmax

**Prompt:** Implement `stable_log_softmax(x, axis=-1)` from scratch using the log-sum-exp trick and assert it matches `mx.log(mx.softmax(x, axis=-1))` within fp32 epsilon on a hard case where the largest logit is ~100 (naive softmax would overflow to `inf`).

**Constraints:**
- Use only `mx.exp`, `mx.sum`, `mx.log`, `mx.max` — do NOT call `mx.softmax`.
- Subtract `x.max(axis=axis, keepdims=True)` before exp (Requirement 4.5 — complexity).
- Include at least one `assert` comparing against `mx.log(mx.softmax(x))` on a non-pathological input (max ~1).
- Include one additional `assert` showing your implementation is FINITE on an overflow case (max ≈ 100) while the naive formula isn't.
- Call `mx.eval` on any lazy MLX result before converting to a Python scalar for comparison.

**Expected complexity:** O(n) per axis — three reductions (max, sum-of-exp, log) plus one elementwise subtraction.

In [11]:
import mlx.core as mx
import numpy as np

def stable_log_softmax(x: mx.array, axis: int = -1) -> mx.array:
    """Log-softmax via max-subtraction. Equivalent to `x - logsumexp(x)`."""
    # c = max(x) along the reduction axis (kept for broadcast).
    c = mx.max(x, axis=axis, keepdims=True)
    shifted = x - c
    # log Σ exp(shifted) — shifted ≤ 0 so each exp is in (0, 1]. No overflow.
    lse = c + mx.log(mx.sum(mx.exp(shifted), axis=axis, keepdims=True))
    return x - lse

# Sanity: matches built-in on a normal input.
x_ok = mx.random.normal(shape=(4, 10), dtype=mx.float32)
ours = stable_log_softmax(x_ok, axis=-1)
ref = mx.log(mx.softmax(x_ok, axis=-1))
mx.eval(ours, ref)
delta = float(mx.max(mx.abs(ours - ref)).item())
assert delta < 1e-5, f"mismatch vs mx.softmax: max|Δ|={delta}"
print(f"✅ matches mx.log(mx.softmax(x)) within {delta:.2e}")

# Overflow case: logits with max ≈ 100.
# Naive: exp(100) ≈ 2.7e43 — FINE in fp32 (fp32 max ≈ 3.4e38 is close but reachable),
# but exp(104) overflows. So push just over the fp32 edge.
x_hard = mx.array([[0.0, 50.0, 100.0, 105.0]], dtype=mx.float32)
naive = mx.exp(x_hard) / mx.sum(mx.exp(x_hard), axis=-1, keepdims=True)
naive_log = mx.log(naive)
ours_hard = stable_log_softmax(x_hard, axis=-1)
mx.eval(naive_log, ours_hard)

naive_np = np.array(naive_log)
ours_np = np.array(ours_hard)
# Naive must produce at least one non-finite entry; ours must be fully finite.
assert not np.all(np.isfinite(naive_np)), (
    f"expected naive to overflow on hard case, got {naive_np}"
)
assert np.all(np.isfinite(ours_np)), (
    f"stable_log_softmax produced non-finite values: {ours_np}"
)
print(f"✅ overflow-case: naive log-softmax = {naive_np[0].tolist()}")
print(f"   stable_log_softmax          = {ours_np[0].tolist()}")


✅ matches mx.log(mx.softmax(x)) within 4.77e-07
✅ overflow-case: naive log-softmax = [-inf, -inf, nan, nan]
   stable_log_softmax          = [-105.0067138671875, -55.0067138671875, -5.0067138671875, -0.0067138671875]


---

### 📐 Complexity & Systems: softmax(x, axis=-1) on a (B, N) tensor

| Quantity | Formula / Value | Notes |
|---|---|---|
| FLOPs | `~5·B·N elementwise ops: one `max` reduction (B·N), one subtract (B·N), one `exp` (B·N), one `sum` reduction (B·N), one divide (B·N)` | per forward pass |
| Memory | `2·B·N·bytes-per-elem (input + output). No O(N²) materialization per row — softmax is purely elementwise on the normalized axis` | working set |
| Latency (M4 Pro, MLX) | `Memory-bound on modern GPUs: peak ~HBM-bandwidth-limited. On M4 Pro / fp32, ~20–80 μs for B=1, N=32k (vocab-size). Fuses with matmul output in `mx.compile`` | measured, see paired benchmark cell |

💡 **Scaling implication:** Softmax is BANDWIDTH-bound, not compute-bound — halving bytes (fp32→fp16) roughly halves wall-time. This is why bf16 decoders see ~2× softmax speed-up even though the FLOP count is identical.

In [12]:
# Benchmark: softmax over last dim on (8, 4096) fp32 (attention-like shape)
import time
import mlx.core as mx

x_const = mx.random.normal(shape=(8, 4096), dtype=mx.float32)

def f():
    return mx.softmax(x_const, axis=-1)


# Warmup — first few runs exclude compile / allocate overhead
for _ in range(3):
    _y = f()
    mx.eval(_y)

N = 10
t0 = time.perf_counter()
for _ in range(N):
    _y = f()
    mx.eval(_y)
dt_ms = (time.perf_counter() - t0) / N * 1000.0
print(f"softmax over last dim on (8, 4096) fp32 (attention-like shape): {dt_ms:.3f} ms / call  (N={N}, 3 warmups)")


softmax over last dim on (8, 4096) fp32 (attention-like shape): 0.138 ms / call  (N=10, 3 warmups)


---
## 🌡️ Temperature Scaling & Sampling

When generating text, we don't always want the most probable token. **Temperature** controls the randomness:

$$\text{softmax}\left(\frac{\text{logits}}{T}\right)$$

- **T → 0**: Distribution becomes a spike on the top token (greedy, deterministic)
- **T = 1.0**: Original distribution (as the model learned it)
- **T → ∞**: Distribution becomes uniform (completely random)

After temperature scaling, we use **sampling strategies** (top-k, top-p) to pick the next token.

### Effect of Temperature on Probability Distributions

Let's visualize how different temperatures reshape the same logits. Watch how low temperature makes the model confident (peaked) and high temperature makes it uncertain (flat).

In [13]:
import mlx.core as mx
import numpy as np
import matplotlib.pyplot as plt

# Same logits, different temperatures
logits = mx.array([3.5, 2.1, 1.8, 0.5, 0.2, -0.3, -0.8, -1.5])
tokens = ["the", "a", "his", "my", "one", "its", "her", "no"]
temperatures = [0.1, 0.5, 1.0, 2.0, 5.0]

fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True)

for ax, T in zip(axes, temperatures):
    scaled_logits = logits / T
    probs = mx.softmax(scaled_logits, axis=-1)
    mx.eval(probs)
    probs_np = np.array(probs.tolist())
    
    colors = plt.cm.RdYlBu_r(0.2 + 0.6 * probs_np / max(probs_np.max(), 1e-8))
    ax.bar(range(len(tokens)), probs_np, color=colors, edgecolor='gray', linewidth=0.5)
    ax.set_title(f"T = {T}", fontsize=12, fontweight='bold')
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
    ax.set_ylim(0, 1.05)
    
    # Annotate top probability
    top_idx = int(np.argmax(probs_np))
    ax.text(top_idx, probs_np[top_idx] + 0.02, f"{probs_np[top_idx]:.2f}",
            ha='center', fontsize=8, fontweight='bold')

axes[0].set_ylabel("Probability")
plt.suptitle("🌡️ Temperature Scaling: Same Logits, Different Distributions", 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("T=0.1: Almost greedy — 99%+ on top token. Good for factual answers.")
print("T=0.5: Confident but allows some variety. Good default for code generation.")
print("T=1.0: Model's learned distribution. Balanced creativity.")
print("T=2.0: Flatter — more random, more creative, more errors.")
print("T=5.0: Nearly uniform — almost random token selection.")

T=0.1: Almost greedy — 99%+ on top token. Good for factual answers.
T=0.5: Confident but allows some variety. Good default for code generation.
T=1.0: Model's learned distribution. Balanced creativity.
T=2.0: Flatter — more random, more creative, more errors.
T=5.0: Nearly uniform — almost random token selection.


/var/folders/gg/dfm95f2x2llgzc8_3fnnxxzc0000gp/T/ipykernel_99189/3081018942.py:33: UserWarning: Glyph 127777 (\N{THERMOMETER}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/gg/dfm95f2x2llgzc8_3fnnxxzc0000gp/T/ipykernel_99189/3081018942.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Top-k Sampling

Instead of sampling from the full vocabulary (50,000+ tokens), **top-k** restricts choices to the $k$ most probable tokens. This prevents the model from picking extremely unlikely tokens while still allowing creativity.

1. Sort logits descending
2. Keep only the top $k$ values
3. Set everything else to $-\infty$
4. Apply softmax to the filtered logits
5. Sample from the resulting distribution

In [14]:
import mlx.core as mx
import numpy as np

def top_k_sampling(logits: mx.array, k: int, temperature: float = 1.0) -> int:
    """Sample a token index using top-k filtering.
    
    Args:
        logits: shape (vocab_size,) — raw model output
        k: number of top tokens to keep
        temperature: temperature scaling factor
    
    Returns:
        Sampled token index
    """
    # Step 1: Apply temperature
    scaled = logits / temperature                       # shape: (vocab_size,)
    
    # Step 2: Find the k-th largest value as threshold
    sorted_logits = mx.sort(scaled)                     # ascending
    threshold = sorted_logits[-k]                        # k-th largest value
    
    # Step 3: Mask everything below threshold
    mask = scaled >= threshold                           # shape: (vocab_size,)
    filtered = mx.where(mask, scaled, mx.array(float('-inf')))  # shape: (vocab_size,)
    
    # Step 4: Softmax over filtered logits
    probs = mx.softmax(filtered, axis=-1)               # shape: (vocab_size,)
    mx.eval(probs)
    
    # Step 5: Sample from distribution
    probs_np = np.array(probs.tolist())
    token_id = np.random.choice(len(probs_np), p=probs_np)
    return token_id

# Demo: vocabulary of 10 tokens
vocab = ["the", "cat", "sat", "on", "mat", "dog", "ran", "big", "red", "and"]
logits = mx.array([3.2, 2.8, 1.5, 1.0, 0.8, 0.3, 0.1, -0.5, -1.0, -2.0])

print("Top-k=3 sampling (10 draws):")
for i in range(10):
    idx = top_k_sampling(logits, k=3, temperature=0.8)
    print(f"  Draw {i+1}: '{vocab[idx]}' (index {idx})")

print(f"\n💡 Only the top 3 tokens ('the', 'cat', 'sat') can be selected.")
print(f"   Lower-ranked tokens like 'red' or 'and' are impossible.")

Top-k=3 sampling (10 draws):
  Draw 1: 'cat' (index 1)
  Draw 2: 'the' (index 0)
  Draw 3: 'the' (index 0)
  Draw 4: 'the' (index 0)
  Draw 5: 'cat' (index 1)
  Draw 6: 'cat' (index 1)
  Draw 7: 'the' (index 0)
  Draw 8: 'cat' (index 1)
  Draw 9: 'cat' (index 1)
  Draw 10: 'the' (index 0)

💡 Only the top 3 tokens ('the', 'cat', 'sat') can be selected.
   Lower-ranked tokens like 'red' or 'and' are impossible.


### Top-p (Nucleus) Sampling

Top-p is smarter than top-k because it adapts to the distribution shape. Instead of a fixed number of tokens, it keeps the **smallest set of tokens whose cumulative probability exceeds $p$**.

- If the model is very confident (one token has 95% probability), top-p might keep just 1-2 tokens
- If the model is uncertain (many tokens with similar probability), top-p keeps more tokens

This is why top-p is the default in most production LLMs (GPT-4 uses top-p=0.9).

In [15]:
def top_p_sampling(logits: mx.array, p: float = 0.9, temperature: float = 1.0) -> int:
    """Sample a token index using nucleus (top-p) filtering.
    
    Args:
        logits: shape (vocab_size,) — raw model output
        p: cumulative probability threshold (0.0 to 1.0)
        temperature: temperature scaling factor
    
    Returns:
        Sampled token index
    """
    # Step 1: Apply temperature
    scaled = logits / temperature                       # shape: (vocab_size,)
    
    # Step 2: Convert to probabilities
    probs = mx.softmax(scaled, axis=-1)                 # shape: (vocab_size,)
    mx.eval(probs)
    probs_np = np.array(probs.tolist())
    
    # Step 3: Sort by probability (descending)
    sorted_indices = np.argsort(probs_np)[::-1]
    sorted_probs = probs_np[sorted_indices]
    
    # Step 4: Find cumulative probability cutoff
    cumulative = np.cumsum(sorted_probs)
    # Keep tokens up to and including the one that crosses threshold p
    cutoff_idx = np.searchsorted(cumulative, p) + 1
    cutoff_idx = min(cutoff_idx, len(sorted_probs))  # safety clamp
    
    # Step 5: Zero out tokens beyond the nucleus
    nucleus_indices = sorted_indices[:cutoff_idx]
    filtered_probs = np.zeros_like(probs_np)
    filtered_probs[nucleus_indices] = probs_np[nucleus_indices]
    
    # Step 6: Re-normalize and sample
    filtered_probs /= filtered_probs.sum()
    token_id = np.random.choice(len(filtered_probs), p=filtered_probs)
    return token_id

# Demo with same vocabulary
print("Top-p=0.9 sampling (10 draws):")
for i in range(10):
    idx = top_p_sampling(logits, p=0.9, temperature=1.0)
    print(f"  Draw {i+1}: '{vocab[idx]}' (index {idx})")

# Show how many tokens are in the nucleus
probs = mx.softmax(logits, axis=-1)
mx.eval(probs)
probs_np = np.array(probs.tolist())
sorted_p = np.sort(probs_np)[::-1]
cumsum = np.cumsum(sorted_p)
nucleus_size = np.searchsorted(cumsum, 0.9) + 1
print(f"\n💡 Nucleus size for p=0.9: {nucleus_size} tokens (out of {len(vocab)})")
print(f"   Cumulative probs: {[f'{c:.3f}' for c in cumsum[:nucleus_size]]}")

Top-p=0.9 sampling (10 draws):
  Draw 1: 'the' (index 0)
  Draw 2: 'the' (index 0)
  Draw 3: 'sat' (index 2)
  Draw 4: 'cat' (index 1)
  Draw 5: 'cat' (index 1)
  Draw 6: 'mat' (index 4)
  Draw 7: 'sat' (index 2)
  Draw 8: 'the' (index 0)
  Draw 9: 'mat' (index 4)
  Draw 10: 'sat' (index 2)

💡 Nucleus size for p=0.9: 5 tokens (out of 10)
   Cumulative probs: ['0.455', '0.759', '0.842', '0.893', '0.934']


🎯 **Interview tip:** "What's the difference between top-k and top-p?" — Top-k always keeps exactly $k$ tokens regardless of confidence. Top-p adapts: it keeps fewer tokens when the model is confident and more when it's uncertain. Most production systems use top-p (nucleus sampling) because it handles both cases well.

⚠️ **Pitfall:** Setting temperature too low with top-p can cause repetitive, boring text. Setting it too high produces incoherent gibberish. The sweet spot for most tasks: T=0.7, top-p=0.9.

---
## 📊 Cross-Entropy Loss

Cross-entropy loss is the training signal for language models. It measures how far the model's predicted probability distribution is from the true distribution (a one-hot vector for the correct next token).

$$\mathcal{L} = -\sum_{i} y_i \log(\hat{y}_i) = -\log(\hat{y}_{\text{correct}})$$

Since the target $y$ is one-hot, only the probability assigned to the **correct** token matters. Lower loss = higher probability on the right answer.

### 💡 What is cross-entropy loss, intuitively?

Think of it like a **weather forecaster's credibility score**. If the forecaster says "90% chance of rain" and it rains, they get a good score. If they say "10% chance of rain" and it rains, they get a terrible score — they were confident *and wrong*, which is the worst combination.

Cross-entropy loss works the same way for language models. The model predicts a probability for each possible next token. The loss only cares about one thing: *"How much probability did you put on the token that actually came next?"* High probability on the right answer = low loss. Low probability on the right answer = high loss. Being confidently wrong is punished much more harshly than being uncertain.

### Manual Implementation Using Log-Softmax

We combine log and softmax into a single numerically stable operation. The log-softmax trick avoids computing softmax first (which can underflow when taking log of tiny probabilities):

$$\log\text{softmax}(x_i) = x_i - \log\left(\sum_j e^{x_j}\right)$$

Then cross-entropy is just: pick the log-probability at the correct class index and negate it.

In [16]:
import mlx.core as mx
import numpy as np

def log_softmax(logits: mx.array) -> mx.array:
    """Numerically stable log-softmax."""
    max_val = mx.max(logits, axis=-1, keepdims=True)    # shape: (..., 1)
    shifted = logits - max_val                           # subtract max for stability
    log_sum_exp = mx.log(mx.sum(mx.exp(shifted), axis=-1, keepdims=True))  # shape: (..., 1)
    return shifted - log_sum_exp                         # shape: same as logits

def cross_entropy_loss(logits: mx.array, targets: mx.array) -> mx.array:
    """Compute cross-entropy loss.
    
    Args:
        logits: shape (batch, vocab_size) — raw model output
        targets: shape (batch,) — correct token indices
    
    Returns:
        Scalar loss (mean over batch)
    """
    log_probs = log_softmax(logits)                      # shape: (batch, vocab_size)
    
    # Gather log-probability at the correct class for each sample
    # This is the "gather" operation: pick one value per row using target indices
    # mx.take_along_axis is the vectorized way to do this (no Python loop!)
    targets_col = mx.expand_dims(targets, axis=-1)       # shape: (batch, 1)
    correct_log_probs = mx.take_along_axis(
        log_probs, targets_col, axis=-1                  # shape: (batch, 1)
    )
    correct_log_probs = mx.squeeze(correct_log_probs, axis=-1)  # shape: (batch,)
    
    loss = -mx.mean(correct_log_probs)                   # negate and average
    return loss

# Test: 3 samples, vocabulary of 5 tokens
logits = mx.array([
    [2.0, 1.0, 0.1, -1.0, -2.0],   # sample 0: model thinks token 0 is most likely
    [0.5, 0.5, 3.0, 0.5, 0.5],     # sample 1: model thinks token 2 is most likely
    [-1.0, -1.0, -1.0, -1.0, 4.0], # sample 2: model thinks token 4 is most likely
])  # shape: (3, 5)

targets = mx.array([0, 2, 4])  # correct tokens: 0, 2, 4 (model is right!)

loss = cross_entropy_loss(logits, targets)
mx.eval(loss)
print(f"Loss (good predictions): {loss.item():.4f}")
print(f"  → Model assigned high probability to correct tokens")

Loss (good predictions): 0.2572
  → Model assigned high probability to correct tokens


### Good vs Bad Predictions

Let's compare the loss when the model predicts correctly vs when it's completely wrong. This shows why cross-entropy is an effective training signal — bad predictions get heavily penalized.

In [17]:
# Good prediction: model puts high probability on correct token
good_logits = mx.array([[5.0, 0.1, 0.1, 0.1, 0.1]])  # shape: (1, 5)
good_target = mx.array([0])  # correct answer is token 0

# Bad prediction: model puts LOW probability on correct token
bad_logits = mx.array([[-2.0, 3.0, 3.0, 3.0, 3.0]])   # shape: (1, 5)
bad_target = mx.array([0])  # correct answer is still token 0, but model disagrees

# Random prediction: model has no idea (uniform-ish)
random_logits = mx.array([[0.0, 0.0, 0.0, 0.0, 0.0]])  # shape: (1, 5)
random_target = mx.array([0])

good_loss = cross_entropy_loss(good_logits, good_target)
bad_loss = cross_entropy_loss(bad_logits, bad_target)
random_loss = cross_entropy_loss(random_logits, random_target)
mx.eval(good_loss, bad_loss, random_loss)

print("Cross-Entropy Loss Comparison:")
print(f"  ✅ Good prediction:   loss = {good_loss.item():.4f}  (model is confident and correct)")
print(f"  🎲 Random prediction: loss = {random_loss.item():.4f}  (model has no idea — this is ln(5) = {np.log(5):.4f})")
print(f"  ❌ Bad prediction:    loss = {bad_loss.item():.4f}  (model is confident but WRONG)")
print(f"\n💡 Notice: bad predictions are penalized much more heavily than random ones.")
print(f"   This steep penalty is what drives the model to learn quickly.")

Cross-Entropy Loss Comparison:
  ✅ Good prediction:   loss = 0.0294  (model is confident and correct)
  🎲 Random prediction: loss = 1.6094  (model has no idea — this is ln(5) = 1.6094)
  ❌ Bad prediction:    loss = 6.3880  (model is confident but WRONG)

💡 Notice: bad predictions are penalized much more heavily than random ones.
   This steep penalty is what drives the model to learn quickly.


### Perplexity: Making Loss Interpretable

Raw cross-entropy loss is hard to interpret. **Perplexity** converts it to an intuitive measure:

$$\text{Perplexity} = e^{\text{loss}}$$

Perplexity answers: **"How many tokens is the model effectively choosing between?"**

- Perplexity = 1: Perfect prediction (model always knows the next token)
- Perplexity = 10: Model is choosing between ~10 equally likely tokens
- Perplexity = 50,000: Model has no idea (random over entire vocabulary)

🎯 **Interview reference:** GPT-4 level perplexity is roughly **10–15** on common benchmarks (like WikiText-103). This means the model narrows down the next token to about 10–15 candidates on average — remarkably good given vocabularies of 100K+ tokens.

In [18]:
import math

# Convert our losses to perplexity
good_ppl = math.exp(good_loss.item())
random_ppl = math.exp(random_loss.item())
bad_ppl = math.exp(bad_loss.item())

print("Perplexity = exp(loss) = 'effective vocabulary confusion'")
print(f"{'Prediction':<20} {'Loss':>8} {'Perplexity':>12} {'Interpretation'}")
print("-" * 70)
print(f"{'Good':.<20} {good_loss.item():>8.4f} {good_ppl:>12.2f}   Model choosing between ~{good_ppl:.0f} tokens")
print(f"{'Random':.<20} {random_loss.item():>8.4f} {random_ppl:>12.2f}   Model choosing between ~{random_ppl:.0f} tokens (= vocab size)")
print(f"{'Bad':.<20} {bad_loss.item():>8.4f} {bad_ppl:>12.2f}   Model is confused and wrong")

print(f"\n🎯 GPT-4 level perplexity reference:")
print(f"   ~ 10-15 on WikiText-103 (choosing between ~10-15 tokens on average)")
print(f"   ~ 8-12 on common English text")
print(f"   This is remarkable given a vocabulary of 100K+ tokens!")

print(f"\n💡 Rule of thumb for training:")
print(f"   Perplexity = vocab_size ({5} here) means the model hasn't learned anything yet.")
print(f"   Watch perplexity drop during training — it's the clearest signal of learning.")

Perplexity = exp(loss) = 'effective vocabulary confusion'
Prediction               Loss   Perplexity Interpretation
----------------------------------------------------------------------
Good................   0.0294         1.03   Model choosing between ~1 tokens
Random..............   1.6094         5.00   Model choosing between ~5 tokens (= vocab size)
Bad.................   6.3880       594.65   Model is confused and wrong

🎯 GPT-4 level perplexity reference:
   ~ 10-15 on WikiText-103 (choosing between ~10-15 tokens on average)
   ~ 8-12 on common English text
   This is remarkable given a vocabulary of 100K+ tokens!

💡 Rule of thumb for training:
   Perplexity = vocab_size (5 here) means the model hasn't learned anything yet.
   Watch perplexity drop during training — it's the clearest signal of learning.


---

### 🎯 Interview Question nb02-q04  ·  [core]  ·  mle, research_engineer, systems_engineer

**Q:** For loss `L = -log softmax(logits)_y` (cross-entropy with one-hot target `y`), derive ∂L/∂logits. Explain why production kernels NEVER multiply explicitly by the softmax Jacobian.

<details>
<summary>Key points in a strong answer</summary>

- Chain rule: ∂L/∂logits = (∂L/∂s) · (∂s/∂logits), where s = softmax(logits).
- ∂L/∂s is a one-hot vector: -1/s_y at position y, zero elsewhere.
- Multiplying by J = diag(s) - ssᵀ and simplifying gives the famous closed form: ∂L/∂logits = s - y  (s is the softmax output, y is the one-hot target).
- No n×n Jacobian ever appears in the final formula — it cancels algebraically. Cost drops from O(n²) to O(n).
- Numerical bonus: computed via log_softmax, the gradient is `exp(log_softmax) - one_hot(y)` — avoids computing exp twice and stays fp16/bf16-safe.
- Every production CE+softmax kernel (vLLM, TRT-LLM, MLX-LM, PyTorch's `cross_entropy`) exploits this fusion; hand-rolled 'softmax then CE' on logits is both slower and less stable.
</details>

> ⚠️ **Trap:** Computing softmax then CE separately and backpropping through both layers — this both wastes FLOPs (the Jacobian-target-onehot cancellation is lost) and introduces a log-of-exp rounding pair that fp16 can't tolerate.
>
> 📚 **References:** https://eli.thegreenplace.net/2016/the-softmax-function-and-its-derivative, https://cs231n.github.io/linear-classify/

---

### 🧑‍💻 Whiteboard Challenge: Derive softmax Jacobian and validate via `mx.grad`

**Prompt:** Implement `softmax_jacobian(x)` returning the n×n closed-form Jacobian J = diag(s) - s·sᵀ where s = softmax(x). Validate it against MLX's autodiff by comparing `J @ v` to the vector-Jacobian product obtained from `mx.grad`.

**Constraints:**
- `x` is a 1-D MLX array of length n; return an (n, n) MLX array.
- Use only `mx.softmax`, `mx.diag`, `mx.outer` (or broadcasting equivalents) — no Python loops over n.
- Compare `J @ v` against `(∂(vᵀ·s)/∂x)` obtained via `mx.grad`. This is the vJp identity the autograd engine uses.
- Assert max-abs-difference < 1e-5 on a random input and a random projection vector `v`.
- Call `mx.eval` on all lazy results before asserting.

**Expected complexity:** O(n²) time & memory for the dense Jacobian; the vJp on the right-hand side is O(n).

In [19]:
import mlx.core as mx

def softmax_jacobian(x: mx.array) -> mx.array:
    """Closed-form Jacobian of softmax: J = diag(s) - s·sᵀ."""
    assert x.ndim == 1, f"expected 1-D input, got shape {x.shape}"
    s = mx.softmax(x, axis=-1)
    # diag(s) - outer(s, s). Broadcasting gives the outer product.
    J = mx.diag(s) - s[:, None] * s[None, :]
    return J

# Random test case.
mx.random.seed(0)
n = 16
x = mx.random.normal(shape=(n,), dtype=mx.float32)
v = mx.random.normal(shape=(n,), dtype=mx.float32)

# Closed-form J @ v.
J = softmax_jacobian(x)
jv_closed = J @ v
mx.eval(jv_closed)

# Autograd: grad of (v · softmax(x)) w.r.t. x == Jᵀ @ v == J @ v (J symmetric).
def fn(x_):
    s_ = mx.softmax(x_, axis=-1)
    return mx.sum(v * s_)

jv_autograd = mx.grad(fn)(x)
mx.eval(jv_autograd)

delta = float(mx.max(mx.abs(jv_closed - jv_autograd)).item())
assert delta < 1e-5, f"closed-form disagrees with mx.grad: max|Δ|={delta}"
print(f"✅ softmax_jacobian matches mx.grad within {delta:.2e}")

# Bonus: verify symmetry and row-sum-to-zero (singularity) properties.
assert float(mx.max(mx.abs(J - J.T)).item()) < 1e-6, "J should be symmetric"
assert float(mx.max(mx.abs(mx.sum(J, axis=-1))).item()) < 1e-6, (
    "J rows should sum to 0 (softmax shift-invariance)"
)
print("✅ J is symmetric and each row sums to 0 (softmax shift-invariance)")


✅ softmax_jacobian matches mx.grad within 2.98e-08
✅ J is symmetric and each row sums to 0 (softmax shift-invariance)


---

### 📐 Complexity & Systems: cross-entropy loss over (B, V) logits with integer targets

| Quantity | Formula / Value | Notes |
|---|---|---|
| FLOPs | `~4·B·V (log-softmax over V) + B (gather + negate). Softmax Jacobian is NEVER materialized — `∂L/∂logits = s - onehot(y)`` | per forward pass |
| Memory | `Forward: 2·B·V bytes working set. Backward: B·V bytes for the gradient (same shape as logits). With typical LLaMA vocab V=128k and batch=8, that's ~4 MB in fp32 per token position` | working set |
| Latency (M4 Pro, MLX) | `Dominated by the log-softmax reduction: ~BANDWIDTH-limited. For B=8, V=32k fp32 on M4 Pro: ~200–600 μs. Fused kernels (vLLM, MLX-LM) cut this by ~2×` | measured, see paired benchmark cell |

💡 **Scaling implication:** At V ≈ 128k and sequence length L ≈ 4k, the per-token CE loss dominates the decoder tail; this is why frontier trainers shard the vocab across TP ranks and compute the loss in tensor-parallel.

In [20]:
# Benchmark: cross-entropy on (B=8, V=32000) logits + int32 targets
import time
import mlx.core as mx

B, V = 8, 32_000
logits_const = mx.random.normal(shape=(B, V), dtype=mx.float32)
targets_const = mx.random.randint(0, V, shape=(B,))

def f():
    # Numerically stable CE: log_softmax then gather target logit.
    ls = logits_const - mx.logsumexp(logits_const, axis=-1, keepdims=True)
    # Gather the target log-prob per row, then negate and mean.
    idx = mx.arange(B)
    return -mx.mean(ls[idx, targets_const])


# Warmup — first few runs exclude compile / allocate overhead
for _ in range(3):
    _y = f()
    mx.eval(_y)

N = 10
t0 = time.perf_counter()
for _ in range(N):
    _y = f()
    mx.eval(_y)
dt_ms = (time.perf_counter() - t0) / N * 1000.0
print(f"cross-entropy on (B=8, V=32000) logits + int32 targets: {dt_ms:.3f} ms / call  (N={N}, 3 warmups)")


cross-entropy on (B=8, V=32000) logits + int32 targets: 0.616 ms / call  (N=10, 3 warmups)


---

### 🛠️ Failure Modes & Debugging: NaN losses, `inf` softmax outputs, or `Infinity` KL — the three classic numerical traps of the math-layer

**Root causes (ranked by frequency):**
1. Softmax overflow: un-shifted `exp(logits)` on logits where one entry > ~88.7 (bf16) or ~11.0 (fp16) → `inf / inf` → NaN.
2. Empty batch: cross-entropy over `(0, V)` logits hits a `0/0` in the mean; many kernels happily return NaN instead of raising.
3. Zero-probability bin in KL(p ‖ q): any position with q_i == 0 but p_i > 0 produces `log(0) = -inf` → the divergence is infinite. Fix by smoothing q (add ε then renormalize) or by masking out positions where p_i == 0.

**Diagnostic code below reproduces the symptom then shows the fix:**

In [21]:
# Reproduce each symptom, then show the fix.
import mlx.core as mx
import numpy as np

# -- Symptom 1: softmax overflow -----------------------------------
bad_logits = mx.array([0.0, 50.0, 100.0, 200.0], dtype=mx.float32)
# Naive: exp(200) ≈ 7e86 — overflows fp32 (max ~3.4e38).
naive_num = mx.exp(bad_logits)
mx.eval(naive_num)
print(f"naive exp:       {np.array(naive_num).tolist()}")  # inf
# Fix: subtract the max before exp.
shifted = bad_logits - mx.max(bad_logits)
stable_num = mx.exp(shifted)
stable_softmax = stable_num / mx.sum(stable_num)
mx.eval(stable_softmax)
print(f"stable softmax:  {np.array(stable_softmax).tolist()}")
assert np.all(np.isfinite(np.array(stable_softmax))), "stable softmax must be finite"

# -- Symptom 2: empty-batch cross-entropy --------------------------
empty_logits = mx.zeros((0, 5), dtype=mx.float32)
empty_targets = mx.zeros((0,), dtype=mx.int32)
# Defensive wrapper — return 0.0 for empty batch instead of NaN.
def safe_cross_entropy(logits: mx.array, targets: mx.array) -> mx.array:
    if logits.shape[0] == 0:
        return mx.array(0.0, dtype=mx.float32)
    ls = logits - mx.logsumexp(logits, axis=-1, keepdims=True)
    idx = mx.arange(logits.shape[0])
    return -mx.mean(ls[idx, targets])
safe_loss = safe_cross_entropy(empty_logits, empty_targets)
mx.eval(safe_loss)
assert float(safe_loss.item()) == 0.0, f"expected 0.0, got {safe_loss.item()}"
print(f"safe CE on empty batch: {safe_loss.item()}")

# -- Symptom 3: zero-probability bin in KL -------------------------
p = mx.array([0.5, 0.5, 0.0])  # third bin has zero mass
q = mx.array([0.1, 0.1, 0.8])  # full support
# Forward KL is FINE: the p_i=0 bin contributes 0·log(0/0.8) := 0.
# Reverse KL(q || p) blows up: q has mass on bin 3 but p_i=0 there.
def kl(a: mx.array, b: mx.array, eps: float = 0.0) -> mx.array:
    b_safe = b + eps
    b_safe = b_safe / mx.sum(b_safe)  # renormalize after smoothing
    # Mask 0·log(0) := 0 to follow the convention.
    term = mx.where(a > 0, a * (mx.log(a) - mx.log(b_safe)), mx.zeros_like(a))
    return mx.sum(term)
kl_pq_naive = kl(p, q, eps=0.0)       # forward: finite
kl_qp_naive = kl(q, p, eps=0.0)       # reverse: inf because p has a 0
mx.eval(kl_pq_naive, kl_qp_naive)
print(f"KL(p||q) naive: {float(kl_pq_naive.item()):.4f}  (finite)")
kl_qp_naive_v = float(kl_qp_naive.item())
print(f"KL(q||p) naive: {kl_qp_naive_v}  ← inf, as expected")
assert not np.isfinite(kl_qp_naive_v), "reverse KL with zero bin should be inf"
# Fix: smooth `p` with a small ε so log(p) is always finite.
kl_qp_smoothed = kl(q, p, eps=1e-8)
mx.eval(kl_qp_smoothed)
assert np.isfinite(float(kl_qp_smoothed.item())), "ε-smoothed KL must be finite"
print(f"KL(q||p) smoothed (ε=1e-8): {float(kl_qp_smoothed.item()):.4f}")


naive exp:       [1.0, 5.1847060206155e+21, inf, inf]
stable softmax:  [0.0, 0.0, 0.0, 1.0]
safe CE on empty batch: 0.0
KL(p||q) naive: 1.6094  (finite)
KL(q||p) naive: inf  ← inf, as expected


KL(q||p) smoothed (ε=1e-8): 14.2361


## 🧪 Try It Yourself

1. **Dot product by hand**: Compute the dot product of [1, 2, 3] and [4, 5, 6] by hand, then verify with `mx.sum(mx.array([1,2,3]) * mx.array([4,5,6]))`.\n\n2. **Softmax exploration**: Apply softmax to [1, 2, 3] and [10, 20, 30]. What happens when the values get larger? Why?\n\n3. **Cross-entropy experiment**: If the true label is class 2 (out of 3 classes), what predicted probability distribution gives the lowest cross-entropy loss?

In [22]:
# 🧪 EXERCISE: Compute softmax by hand, then verify with MLX
# Fill in the blanks (???) and run to check!

import mlx.core as mx
import numpy as np

# Step 1: Start with raw scores
scores = [2.0, 1.0, 0.1]
print(f"Raw scores: {scores}")

# Step 2: Compute e^x for each score
# e ≈ 2.718, so e^2 ≈ 7.389, e^1 ≈ 2.718, e^0.1 ≈ 1.105
exp_scores = [np.exp(s) for s in scores]
print(f"After exp(): {[round(e, 3) for e in exp_scores]}")

# Step 3: Sum them up
total = sum(exp_scores)
print(f"Sum of exp scores: {total:.3f}")

# Step 4: Divide each by the sum → probabilities!
softmax_result = [e / total for e in exp_scores]
print(f"Softmax (by hand): {[round(s, 4) for s in softmax_result]}")
print(f"Sum of probabilities: {sum(softmax_result):.4f}  (should be 1.0)")

# Step 5: Verify with MLX
mlx_result = mx.softmax(mx.array(scores)).tolist()
print(f"Softmax (MLX):     {[round(s, 4) for s in mlx_result]}")
print(f"\n✅ They match! Softmax = exp(x) / sum(exp(x)) — turns any numbers into probabilities.")

Raw scores: [2.0, 1.0, 0.1]
After exp(): [np.float64(7.389), np.float64(2.718), np.float64(1.105)]
Sum of exp scores: 11.213
Softmax (by hand): [np.float64(0.659), np.float64(0.2424), np.float64(0.0986)]
Sum of probabilities: 1.0000  (should be 1.0)
Softmax (MLX):     [0.659, 0.2424, 0.0986]

✅ They match! Softmax = exp(x) / sum(exp(x)) — turns any numbers into probabilities.


---
## 🏁 Summary

You now have hands-on understanding of the five math operations that power every transformer:

| Operation | What It Does | Where It's Used |
|---|---|---|
| **Dot product** | Measures vector similarity | Attention scores (Q·K) |
| **Matrix multiply** | Transforms embeddings | Every linear layer, Q/K/V projections |
| **Softmax** | Logits → probabilities | Attention weights, output layer |
| **Temperature + sampling** | Controls randomness | Text generation (top-k, top-p) |
| **Cross-entropy loss** | Measures prediction error | Training signal |

🎯 **Key takeaway for interviews:** A transformer is fundamentally just matrix multiplications and softmax, repeated many times. Everything else (attention, FFN, normalization) is built from these primitives.

**Next up:** Notebook 03 — Tokenization (how text becomes numbers the model can process).

---
## 🔑 Key Takeaways

1. **Dot product = similarity detector.** In attention, `query @ key` measures how relevant one token is to another. Higher dot product → more attention. Cosine similarity normalizes this to remove magnitude bias.

2. **Matrix multiplication = the workhorse.** Every linear layer, every Q/K/V projection, every FFN — it's all matmul. ~99% of a transformer's compute is matrix multiplication. Larger matrices utilize the GPU better.

3. **Softmax = score-to-probability converter.** It takes arbitrary numbers and produces a valid probability distribution (all positive, sums to 1). The max-subtraction trick is essential for numerical stability — never skip it.

4. **Temperature controls the confidence dial.** Low temperature (→0) makes the model pick the top token almost deterministically. High temperature (→∞) makes it nearly random. Top-p sampling adapts to the model's confidence level, which is why production systems prefer it over top-k.

5. **Cross-entropy loss = the learning signal.** It only cares about one thing: how much probability did you assign to the correct next token? Being confidently wrong is punished exponentially harder than being uncertain. Perplexity (= e^loss) tells you "how many tokens is the model effectively choosing between."

6. **Numerical stability is not optional.** Log-softmax fusion, max-subtraction in softmax, and proper initialization (Xavier/Kaiming) are the difference between a model that trains and one that produces NaN after 100 steps.

---

### 🎯 Interview Question nb02-q05  ·  [stretch]  ·  research_engineer, systems_engineer

**Q:** State the three defining properties of KL(p ‖ q), give a concrete example where KL(p ‖ q) ≠ KL(q ‖ p), and explain why RLHF/DPO uses the 'reverse' direction.

<details>
<summary>Key points in a strong answer</summary>

- Property 1 — non-negative: KL(p ‖ q) ≥ 0 (Gibbs' inequality, from Jensen applied to the convex −log).
- Property 2 — zero iff equal: KL(p ‖ q) = 0 ⇔ p = q almost everywhere.
- Property 3 — asymmetric: KL is NOT a metric (no triangle inequality, not symmetric); it's a *premetric*.
- Asymmetry example: p = [0.9, 0.1], q = [0.1, 0.9] → KL(p‖q) = 0.9·log(9) + 0.1·log(1/9) ≈ 1.76 nats, KL(q‖p) ≈ 1.76 nats (symmetric here by luck); try p=[1,0,0], q=[1/3,1/3,1/3]: KL(p‖q)=log(3)≈1.10, KL(q‖p)=∞ because q has mass where p has none.
- Zero-probability bin is the killer: KL(p‖q) blows up when q_i=0 but p_i>0 — this is why 'sample from p, score under q' requires q to have FULL support over p.
- RLHF / DPO use KL(π_policy ‖ π_ref) (policy on the left) so the loss is finite: the reference model is trained first and has full support; the policy just needs to stay close.
- Forward KL (p‖q) is 'mean-seeking' (zero-avoiding); reverse KL (q‖p) is 'mode-seeking' (zero-forcing) — MLE training minimizes forward KL of data ‖ model; RLHF minimizes reverse KL of policy ‖ reference.
</details>

> ⚠️ **Trap:** Calling KL 'a distance' — it's not. It fails BOTH symmetry and the triangle inequality; it's only a divergence. Candidates who write `kl(p,q) == kl(q,p)` get filtered out.
>
> 📚 **References:** https://en.wikipedia.org/wiki/Kullback%E2%80%93Leibler_divergence, https://arxiv.org/abs/2305.18290

---

### 🎯 Interview Question nb02-q06  ·  [stretch]  ·  research_engineer, systems_engineer

**Q:** Connect the dots: what do entropy H(X), cross-entropy H(p, q), KL(p ‖ q), and model perplexity all have to do with each other? Give the exact identity.

<details>
<summary>Key points in a strong answer</summary>

- Entropy: H(p) = -Σ p_i log p_i. The self-information of the data distribution; in nats if log=ln, bits if log₂.
- Cross-entropy: H(p, q) = -Σ p_i log q_i. The avg # bits/nats needed to encode samples from p using a code optimized for q.
- The fundamental identity: H(p, q) = H(p) + KL(p ‖ q). Cross-entropy EQUALS entropy PLUS KL.
- LLM training minimizes H(p_data, p_model); since H(p_data) is fixed, this is exactly minimizing KL(p_data ‖ p_model).
- Perplexity = exp(cross-entropy per token). 'The model is choosing among PPL tokens on average' — a PPL of 10 means the model is as uncertain as if uniformly picking one of 10 options per position.
- Mutual information: I(X; Y) = H(X) - H(X|Y) = KL(p(x,y) ‖ p(x)p(y)). Measures how much knowing Y reduces uncertainty about X; appears in InfoNCE, contrastive losses, and information-bottleneck arguments.
</details>

> ⚠️ **Trap:** Confusing perplexity with cross-entropy — perplexity is the EXPONENTIAL of the per-token CE, not CE itself. A 2× drop in CE is an exponential (not 2×) drop in perplexity.
>
> 📚 **References:** https://en.wikipedia.org/wiki/Cross_entropy, https://en.wikipedia.org/wiki/Mutual_information

---

### 🏭 How Production Systems Handle This — Fused softmax + cross-entropy + Flash-Attention softmax

| System | Mechanism | Notes |
|---|---|---|
| vLLM | Uses PyTorch's `F.cross_entropy` (fused log-softmax + NLL in a single CUDA kernel) for training; inference path uses FlashAttention-2 whose implicit online softmax keeps fp16 safe and avoids materializing the (N × N) attention matrix. | |
| SGLang | Inherits PyTorch's fused CE kernel. Its RadixAttention pre-computes softmax normalization constants once per shared prefix then reuses them — a cache-oriented twist on the LSE trick. | |
| TensorRT-LLM | Builds a single fused kernel `softmax_cross_entropy` at engine-compile time; attention uses FMHA/FMHCA kernels with online softmax baked into the tile loop. No user-visible softmax op exists at runtime. | |
| MLX-LM | `mx.softmax` + `mx.logsumexp` fuse automatically under `mx.compile`; attention uses the Metal-native SDPA kernel which, like FA-2, streams softmax tile-by-tile — bf16-safe on M-series. | |

🎯 **Interview tip:** Know at least one concrete trade-off per row.

---

### 🔭 Frontier Context (Numerical tricks in modern attention + softmax)

**Paper trail:**
1. FlashAttention-2: Faster Attention with Better Parallelism (Dao, 2023) (2023) — Formalized the online-softmax tile-merge identity and showed 2× wall-clock over FA-1 on A100; baseline for every 2024+ attention kernel including MLX's SDPA.
2. FlashAttention-3: Fast and Accurate Attention with Asynchrony and Low-Precision (Shah et al., 2024) (2024) — Hopper-specific; uses fp8 for attention ops but promotes the softmax ACCUMULATOR (m, ℓ) to fp32 — the same rule DeepSeek-V3 follows for its attention rewrite.
3. DeepSeek-V3 Technical Report (DeepSeek, 2024) (2024) — Documents the 'fp32 accumulator in softmax' rule explicitly as a stability requirement at frontier scale (671 B params, trained in fp8 + bf16). The log-sum-exp tile-merge is un-changed from FA-2.
4. Online Softmax + Streaming LLMs (community, 2024–2025) (2025) — StreamingLLM, Windowed Attention, and sink tokens all reuse the LSE tile-merge so they can discard old KV without re-normalizing softmax on the surviving tokens — the merge formula is the only reason streaming doesn't corrupt probabilities.
5. Gemma 2/3/4 engineering notes on MLX (Apple / community, 2025) (2025) — Port of Gemma to MLX inherits the bf16 softmax + fp32 accumulator rule; `mx.softmax` on M-series auto-promotes the reduction for bf16 inputs to preserve tail precision.

**Current SOTA:** As of late 2025, every frontier attention kernel (FA-2, FA-3, MLX-SDPA, TRT-LLM FMHA, vLLM V1 paged attention) uses the exact log-sum-exp tile-merge formula from FA-2; the only variation is fp8 vs bf16 vs fp16 storage and where exactly the fp32 accumulator lives. Active frontier: softmax-free attention (ReLU² attention, polynomial kernels) for sub-quadratic attention — the trade-off is expressivity vs LSE's provable numerical stability.

## 📜 History & Alternatives

### Mathematical Foundations That Shaped Deep Learning

The math behind modern LLMs draws from centuries of mathematical development. Each breakthrough unlocked a new capability in neural networks.

| Year | Innovation | Mathematician / Team | Key Contribution |
|------|-----------|---------------------|-----------------|
| 1805 | **Least Squares** | Legendre / Gauss | Foundation of optimization — minimizing squared error |
| 1847 | **Gradient Descent** | Cauchy | Iterative optimization by following negative gradients |
| 1901 | **PCA / SVD** | Pearson / Beltrami | Dimensionality reduction — basis for understanding embeddings |
| 1948 | **Information Theory** | Claude Shannon | Entropy, cross-entropy — the loss functions we minimize |
| 1958 | **Perceptron** | Frank Rosenblatt | First trainable neural network — linear classification |
| 1969 | **Backpropagation (concept)** | Bryson & Ho | Efficient gradient computation through computational graphs |
| 1986 | **Backprop for NNs** | Rumelhart, Hinton, Williams | Made training multi-layer networks practical — the algorithm that launched modern deep learning |
| 1990 | **Softmax / Cross-Entropy in NNs** | Bridle; Hinton | Softmax output layers with cross-entropy loss became the standard for classification and language modeling |
| 1997 | **LSTM** | Hochreiter & Schmidhuber | Gating mechanisms to handle vanishing gradients in sequences |
| 2010 | **Xavier Initialization** | Glorot & Bengio | Weight init preserving activation variance across layers — solved the "dying network" problem for sigmoid/tanh |
| 2012 | **AlexNet / ReLU** | Krizhevsky, Sutskever, Hinton | ReLU activation solved vanishing gradients for deep CNNs |
| 2014 | **Attention Mechanism** | Bahdanau, Cho, Bengio | Additive attention for seq2seq — the seed of the transformer revolution |
| 2014 | **Adam Optimizer** | Kingma & Ba | Adaptive learning rates — became the default optimizer |
| 2015 | **Batch Normalization** | Ioffe & Szegedy | Stabilized training of very deep networks by normalizing activations |
| 2015 | **Kaiming Initialization** | He, Zhang, Ren, Sun | Weight init designed for ReLU networks — preserves variance through ReLU nonlinearity |
| 2016 | **Layer Normalization** | Ba, Kiros, Hinton | Normalization for sequence models (no batch dependency) |
| 2017 | **Scaled Dot-Product Attention** | Vaswani et al. | QK^T/√d_k — the core operation of transformers; √d_k scaling is a numerical stability technique |
| 2019 | **RMSNorm** | Zhang & Sennrich | Simplified LayerNorm — just scale, no centering |
| 2020 | **Numerical Stability at Scale** | Various (Megatron-LM, DeepSpeed) | Log-sum-exp tricks, mixed-precision training, loss scaling — essential for training billion-parameter models |

### 💡 The Three Pillars of LLM Math

1. **Linear Algebra**: Matrix multiplication is the core operation — attention (QK^T), FFN layers, embeddings are all matmuls
2. **Calculus / Optimization**: Backpropagation computes gradients; Adam/AdamW navigate the loss landscape
3. **Probability / Information Theory**: Cross-entropy loss, softmax, sampling strategies, KL divergence for alignment

### Why These Specific Topics Matter

| Math Concept | Where It Appears in LLMs | Why It Matters |
|-------------|-------------------------|---------------|
| Matrix multiplication | Every layer | 99% of compute is matmul |
| Softmax | Attention weights, output logits | Converts scores to probabilities |
| Cross-entropy | Training loss | The objective we optimize |
| Chain rule | Backpropagation | How gradients flow through layers |
| Eigenvalues | Attention analysis, PCA | Understanding what attention learns |
| KL divergence | RLHF, DPO, knowledge distillation | Measuring distribution differences |

### ⚠️ Common Numerical Pitfalls

- **Naive softmax overflow**: Computing e^x directly for large x causes Inf. Always subtract the max first: softmax(x) = softmax(x - max(x))
- **Log-of-zero**: Cross-entropy requires log(p) — if p = 0, you get -Inf. Use log-softmax (fused) to avoid this
- **Vanishing gradients with bad init**: Without Xavier/Kaiming initialization, activations either explode or vanish after a few layers
- **Float16 underflow**: Gradients in mixed-precision training can underflow to zero — loss scaling compensates by multiplying loss before backward pass

### ⚡ Computational Perspective

Modern LLMs spend ~99% of their FLOPs on matrix multiplications. A single forward pass of Llama 3.1 70B performs roughly 140 TFLOPs of matmul operations. Understanding how matrices multiply — and how hardware accelerates them (tiling, fusion, quantization) — is the single most important mathematical concept for LLM engineering.

### 🎯 Interview Tip

> *"The softmax function in attention (softmax(QK^T/√d_k)) serves two purposes: it normalizes attention weights to form a valid probability distribution, and the √d_k scaling prevents dot products from growing too large in high dimensions — which would push softmax into saturation where gradients vanish. This scaling is derived from the variance of dot products: if Q and K entries are ~N(0,1), then QK^T entries have variance d_k, so dividing by √d_k restores unit variance."*

---

### 📋 Interview Question Index

| ID | Difficulty | Roles | Question |
|---|---|---|---|
| `nb02-q01` | warmup | mle | What is the difference between numerator-layout and denominator-layout matrix... |
| `nb02-q02` | core | mle, research_engineer | The naive softmax `exp(x) / sum(exp(x))` blows up on attention logits near ma... |
| `nb02-q03` | core | mle, research_engineer, systems_engineer | Derive the Jacobian of softmax(x) ∈ ℝⁿ w.r.t. x ∈ ℝⁿ from scratch. Why does t... |
| `nb02-q04` | core | mle, research_engineer, systems_engineer | For loss `L = -log softmax(logits)_y` (cross-entropy with one-hot target `y`)... |
| `nb02-q05` | stretch | research_engineer, systems_engineer | State the three defining properties of KL(p ‖ q), give a concrete example whe... |
| `nb02-q06` | stretch | research_engineer, systems_engineer | Connect the dots: what do entropy H(X), cross-entropy H(p, q), KL(p ‖ q), and... |
| `nb02-q07` | research | research_engineer, systems_engineer | Flash-Attention computes softmax 'online' — one tile at a time, without mater... |